# Genetic Algorithms for Feature Selection

In [1]:
import numpy as np

from sklearn.datasets import make_classification
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


In [2]:
seed = 10000

np.random.seed(seed)

In [3]:
# Funkcja przystosowania: acc modelu dla danego zestawu zmiennych - lamda * (kara za liczbę zmiennych)

def fitness(chrom, X_tr, y_tr, X_val, y_val, lam):
    idx = np.where(chrom == 1)[0]
    if len(idx) == 0:
        return -1.0

    nb = GaussianNB()
    nb.fit(X_tr[:, idx], y_tr)
    acc = accuracy_score(y_val, nb.predict(X_val[:, idx]))
    return acc - lam * (len(idx) / chrom.size)


In [4]:
# Selekcja turniejowa

def tournament(population, scores, k=3):
    ids = np.random.choice(len(population), k, replace=False)
    return population[ids[np.argmax(scores[ids])]].copy()


In [5]:
# Krzyżowanie równomierne

def uniform_crossover(p1, p2):
    position = np.random.rand(len(p1)) < 0.5
    child = p1.copy()
    child[position] = p2[position]
    return child


In [6]:
# Mutacja

def mutate(child, p):
    flip = np.random.rand(len(child)) < p
    child[flip] = 1 - child[flip]

    if child.sum() == 0:
        child[np.random.randint(len(child))] = 1
        
    return child

In [7]:
def ga_feature_selection(X, y, population_size, generations, mutation_rate, lam):
    
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, stratify=y,  random_state=seed)    # Wydzielenie zbioru walidacyjnego

    d = X.shape[1]
    population = (np.random.rand(population_size, d) < 0.5).astype(int)     # Wygenerowanie populacji początkowej

    best_features = None
    best_acc = 0

    for _ in range(generations):
        scores = np.array([fitness(chrom, X_tr, y_tr, X_val, y_val, lam) for chrom in population])  # fitness każdego osobnika

        i = np.argmax(scores)       # zapis najlepszego acc i zestawu parametrów
        if scores[i] > best_acc:
            best_acc = scores[i]
            best_features = population[i].copy()

        new_population = []
        while len(new_population) < population_size:  # generowanie nowej populacji
            p1 = tournament(population, scores)
            p2 = tournament(population, scores)
            child = uniform_crossover(p1, p2)
            child = mutate(child, mutation_rate)
            new_population.append(child)

        population = np.array(new_population) # podmiana populacji

    return best_features


# Eksperyment

### Porównanie miary accuracy modelu klasyfikacji Naive Bayes przed i po selekcji zmiennych za pomoca algorytmu genetycznego.

### Dane:

In [8]:
X, y = make_classification(         # Problem klasyfikacji binarnej
    n_samples=2000,
    n_features=20,
    n_informative=14,               # 14 zmiennych istotnych
    n_redundant=6,                  # 6 zmiennych zbędnych
    n_classes=2,
    shuffle=False,
    random_state=seed
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y,  random_state=seed
)

#### Istotne zmienne: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

### Przed selekcją zmiennych:

In [9]:
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_all = nb.predict(X_test)
acc_all = accuracy_score(y_test, y_pred_all)

### Po selekcji zmiennych:

In [10]:
# Parametry ewolucji

population_size=40      # Wielkość populacji
generations=60          # Liczba pokoleń
mutation_rate=0.05      # Współczynnik mutacji
lam=0.02                # Współczynnik kary

In [11]:
selection = ga_feature_selection(X_train, y_train, population_size, generations, mutation_rate, lam)

selected_idx = np.where(selection == 1)[0]

nb_select = GaussianNB()
nb_select.fit(X_train[:, selected_idx], y_train)

y_pred_select = nb_select.predict(X_test[:, selected_idx])
acc_select = accuracy_score(y_test, y_pred_select)

### Wyniki

In [12]:
print(f"Accuracy (wszystkie zmienne): {acc_all:.4f}")
print(f"Accuracy (po selekcji GA): {acc_select:.4f}")
print("\n")
print("Wybrane indeksy cech:", selected_idx.tolist())
print("Liczba wybranych cech:", len(selected_idx))

Accuracy (wszystkie zmienne): 0.8467
Accuracy (po selekcji GA): 0.8717


Wybrane indeksy cech: [1, 2, 3, 4, 6, 8, 10, 11, 13, 17]
Liczba wybranych cech: 10
